In [1]:
import os
import sys
from google.colab import drive, userdata

# 1. Mount Cloud Storage Base
print("🔄 Connecting to Google Drive Cluster...")
drive.mount('/content/drive', force_remount=True)

# 2. Extract Secure Access Token from Secrets Manager
print("🔑 Authenticating Git Credentials...")
try:
    token = userdata.get("Github_Token")
except Exception:
    raise ValueError("❌ Please add 'Github_Token' to your Colab Left Sidebar Secrets (🔑 icon) before running.")

# 3. Synchronize Repository
REPO_DIR = "/content/economic-news-sentiment"
if not os.path.exists(REPO_DIR):
    print("📥 Clowning repository down to working machine runtime...")
    !git clone https://{token}@github.com/Nas-Mohd/economic-news-sentiment.git
else:
    print("🔄 Repo folder already exists. Checking current status...")

# 4. Bind Global Git Profiling Rules
!git config --global user.email "anasamohammad.school@gmail.com"
!git config --global user.name "Nas-Mohd"

# 5. Path Python Environment Directly to Repo Tree
sys.path.append(REPO_DIR)

# 6. Sanity Status Check
print("\n🛰️ Current Repository Version Track Tracking Status:")
!git -C {REPO_DIR} status

# 7. Environment Dependency Assurance
!pip install -q transformers[torch] datasets accelerate scikit-learn pandas numpy tqdm

🔄 Connecting to Google Drive Cluster...
Mounted at /content/drive
🔑 Authenticating Git Credentials...
📥 Clowning repository down to working machine runtime...
Cloning into 'economic-news-sentiment'...
remote: Enumerating objects: 572, done.
remote: Counting objects: 100% (189/189), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 572 (delta 116), reused 107 (delta 52), pack-reused 383 (from 1)
Receiving objects: 100% (572/572), 1.50 MiB | 5.04 MiB/s, done.
Resolving deltas: 100% (359/359), done.

🛰️ Current Repository Version Track Tracking Status:
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Point directly to your repository's sentiment data path
SENTIMENT_DATA_PATH = "/content/economic-news-sentiment/training/data/finbert_absa_training_ready.csv"

DOMAIN_LABELS = [
    "Monetary_Financial", "Inflation_Prices", "Real_Economic_Activity",
    "Labor_Consumption", "Fiscal_Government", "External_Sector"
]

print(f"🔄 Loading raw sentiment dataset from: {SENTIMENT_DATA_PATH}")
if not os.path.exists(SENTIMENT_DATA_PATH):
    raise FileNotFoundError(f"❌ Could not locate sentiment file at {SENTIMENT_DATA_PATH}!")

raw_df = pd.read_csv(SENTIMENT_DATA_PATH)

def transform_to_aspect_sentiment(df_raw):
    aspect_rows = []
    for _, row in df_raw.iterrows():
        for aspect in DOMAIN_LABELS:
            # Check the numerical cell value
            score = int(row[aspect])

            # 0 = Abstain (Skip). 1 = Positive, 2 = Negative, 3 = Neutral.
            if score in [1, 2, 3]:
                # Map scale from 1-3 to standard 0-2 index structure for Hugging Face CrossEntropy
                hf_label = score - 1

                # 2. Use regex to swap ALL underscores (single or multiple) with a clean space
                clean_aspect = re.sub(r'_+', ' ', aspect).strip()

                aspect_rows.append({
                    "text": str(row["text"]),
                    "aspect": clean_aspect, # Clean up formatting for attention text pairs
                    "label": hf_label
                })
    return pd.DataFrame(aspect_rows)

# 1. Transform the matrix into single exploded context pairs
exploded_df = transform_to_aspect_sentiment(raw_df)
print(f"💥 Exploded dataset into {len(exploded_df)} distinct aspect-sentiment records.")

# 2. Perform a split directly on this clean target dataset (e.g., 90% Train, 10% Val)
train_df_sentiment, val_df_sentiment = train_test_split(
    exploded_df,
    test_size=0.10,
    random_state=42,
    stratify=exploded_df['label'] # Ensures emotional class balance across sets
)

print(f"• Training Pairs:   {len(train_df_sentiment)}")
print(f"• Validation Pairs: {len(val_df_sentiment)}")
print("\n--- Sample Processed Entry ---")
print(train_df_sentiment.head(1).to_dict(orient='records'))

🔄 Loading raw sentiment dataset from: /content/economic-news-sentiment/training/data/finbert_absa_training_ready.csv
💥 Exploded dataset into 3750 distinct aspect-sentiment records.
• Training Pairs:   3375
• Validation Pairs: 375

--- Sample Processed Entry ---
[{'text': '- Global oil use is expected to fall over 2026 as a whole as a result of the Hormuz closure and the destruction of energy infrastructure across the Gulf from retaliatory Iranian attacks.', 'aspect': 'Real Economic Activity', 'label': 1}]


In [3]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "ProsusAI/finbert"
print(f"📥 Loading foundational {MODEL_NAME} tokenizer asset...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class AspectSentimentDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # This formatting strategy forces: [CLS] text [SEP] aspect [SEP]
        inputs = self.tokenizer(
            text=str(row['text']),
            text_pair=str(row['aspect']),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        item = {key: val.squeeze(0) for key, val in inputs.items()}
        item['labels'] = torch.tensor(int(row['label']), dtype=torch.long)
        return item

# Construct live PyTorch datasets
train_dataset = AspectSentimentDataset(train_df_sentiment, tokenizer)
val_dataset = AspectSentimentDataset(val_df_sentiment, tokenizer)
print("📌 PyTorch dataset transformation maps fully compiled and cached in memory.")

📥 Loading foundational ProsusAI/finbert tokenizer asset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

📌 PyTorch dataset transformation maps fully compiled and cached in memory.


In [4]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    """
    Computes system operational performance over standard 3-class array heads
    Mapping: 0 -> Positive, 1 -> Negative, 2 -> Neutral
    """
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    preds = np.argmax(logits, axis=1)

    # Calculate Macro F1 to evaluate balanced focus over all 3 sentiments
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "macro_f1": macro_f1
    }

print("🎯 Evaluation metric computation engines safely attached.")

🎯 Evaluation metric computation engines safely attached.


In [5]:
import os
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Route your checkpoint directory cleanly inside your checked-out local repo path
REPO_CHECKPOINT_DIR = "/content/economic-news-sentiment/finbert_checkpoints"
os.makedirs(REPO_CHECKPOINT_DIR, exist_ok=True)

print("🔄 Loading native FinBERT baseline weights...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    ignore_mismatched_sizes=True # Safely swaps out the default classification layer mapping
)

training_args = TrainingArguments(
    output_dir=REPO_CHECKPOINT_DIR,    # Output updates drop straight into local repo path
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,                # Keeps folder neat by discarding legacy checkpoints instantly
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=15,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# Automate Checkpoint Recovery Lookups within the Repo File System
checkpoint_flag = False
if os.path.exists(REPO_CHECKPOINT_DIR) and len(os.listdir(REPO_CHECKPOINT_DIR)) > 0:
    for sub in os.listdir(REPO_CHECKPOINT_DIR):
        if "checkpoint" in sub:
            checkpoint_flag = True

print("\n🏁 Setup Complete!")
if checkpoint_flag:
    print("➡️ Existing checkpoint found inside Git repo structure. Training states will resume safely!")
else:
    print("➡️ No local checkpoint found. Preparing a completely clean training pass.")

🔄 Loading native FinBERT baseline weights...


pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🏁 Setup Complete!
➡️ No local checkpoint found. Preparing a completely clean training pass.


In [6]:
print("🚀 Launching FinBERT Aspect Sentiment fine-tuning loops...")

# Start training, allowing seamless crash recovery execution if flags were raised
train_metrics = trainer.train(resume_from_checkpoint=checkpoint_flag)

print("\n🎉 Fine-tuning sequence executed successfully!")
print(train_metrics)

🚀 Launching FinBERT Aspect Sentiment fine-tuning loops...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.759275,0.665919,0.714667,0.592464
2,0.577913,0.645904,0.728000,0.664030
3,0.459878,0.696630,0.738667,0.675943
4,0.308037,0.752370,0.752000,0.683429
5,0.194629,0.792830,0.746667,0.687253


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


🎉 Fine-tuning sequence executed successfully!
TrainOutput(global_step=1055, training_loss=0.4612340341812061, metrics={'train_runtime': 490.3634, 'train_samples_per_second': 34.413, 'train_steps_per_second': 2.151, 'total_flos': 1110009731040000.0, 'train_loss': 0.4612340341812061, 'epoch': 5.0})


In [8]:
FINAL_DRIVE_OUT = "/content/drive/MyDrive/economic_news_project/final_finbert_aspect_sentiment"
os.makedirs(FINAL_DRIVE_OUT, exist_ok=True)

print(f"💾 Exporting fully optimized best weights tracking state directly to Drive container...")

# Save clean tracking layers only—excluding massive local optimizer artifacts from Drive!
trainer.model.save_pretrained(FINAL_DRIVE_OUT)
tokenizer.save_pretrained(FINAL_DRIVE_OUT)

print("="*65)
print(f"✅ PIPELINE CONCLUDED!")
print(f"Your optimized aspect sentiment model is permanently stored in cloud space at:")
print(f"📂 {FINAL_DRIVE_OUT}")
print("="*65)

💾 Exporting fully optimized best weights tracking state directly to Drive container...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ PIPELINE CONCLUDED!
Your optimized aspect sentiment model is permanently stored in cloud space at:
📂 /content/drive/MyDrive/economic_news_project/final_finbert_aspect_sentiment


In [9]:
import os
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Configured to point directly to your pristine Google Drive export directory
MODEL_DRIVE_PATH = "/content/drive/MyDrive/economic_news_project/final_finbert_aspect_sentiment"

DOMAIN_LABELS = [
    "Monetary Financial",
    "Inflation Prices",
    "Real Economic_Activity",
    "Labor Consumption",
    "Fiscal Government",
    "External Sector"
]

SENTIMENT_MAP = {0: "🟢 Positive", 1: "🔴 Negative", 2: "🟡 Neutral"}

print("🔄 Initializing interactive test pipeline. Loading weights from Google Drive...")
if not os.path.exists(MODEL_DRIVE_PATH):
    raise FileNotFoundError(f"❌ Could not find fine-tuned weights at {MODEL_DRIVE_PATH}. Did you run Cell 7?")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DRIVE_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DRIVE_PATH)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()
print(f"✅ Live Model Engine deployed successfully on: [{device.upper()}]")
print("="*70)

# 2. Interactive Loop Execution Entry
while True:
    print("\n" + "-"*50)
    user_sentence = input("✍️ Enter an economic news sentence (or type 'exit'): ").strip()
    if user_sentence.lower() == 'exit':
        print("👋 Exiting sandbox loop. Happy coding!")
        break
    if not user_sentence:
        continue

    print("\n📋 Select Target Macro Aspect Group:")
    for i, domain in enumerate(DOMAIN_LABELS):
        print(f"  [{i}] {domain}")

    try:
        selection = input("\n🔢 Enter Aspect Number (0-5): ").strip()
        aspect_idx = int(selection)
        if aspect_idx < 0 or aspect_idx >= len(DOMAIN_LABELS):
            print("❌ Invalid number choice! Resetting entry frame...")
            continue
    except ValueError:
        print("❌ Please enter a valid numerical index number!")
        continue

    chosen_aspect = DOMAIN_LABELS[aspect_idx]

    # 3. Apply your native Token-Appending Syntax Strategy: [CLS] sentence [SEP] aspect [SEP]
    inputs = tokenizer(
        text=user_sentence,
        text_pair=chosen_aspect,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    # 4. Extract Logit Vectors and Map via Softmax Distribution Scaling
    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]

    winning_idx = np.argmax(probabilities)
    winning_label = SENTIMENT_MAP[winning_idx]
    winning_conf = probabilities[winning_idx]

    # 5. Output Multi-Class Confidence Distribution Graph
    print("\n📊 ==================== MODEL ANALYSIS ====================")
    print(f"📝 Text  : \"{user_sentence}\"")
    print(f"🎯 Aspect: {chosen_aspect}")
    print(f"🏆 Result: {winning_label} ({winning_conf*100:.2f}% Confidence)\n")
    print("📈 Full Probability Breakdown Matrix:")

    for idx, name in SENTIMENT_MAP.items():
        prob = probabilities[idx]
        bar_length = int(prob * 30)
        visual_bar = "█" * bar_length + "░" * (30 - bar_length)
        print(f"  {name:<11} | {visual_bar} | {prob*100:6.2f}%")
    print("============================================================")

🔄 Initializing interactive test pipeline. Loading weights from Google Drive...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Live Model Engine deployed successfully on: [CUDA]

--------------------------------------------------
✍️ Enter an economic news sentence (or type 'exit'): High interest rates continue to damp consumer demand, dragging down retail sales margins throughout the quarter

📋 Select Target Macro Aspect Group:
  [0] Monetary Financial
  [1] Inflation Prices
  [2] Real Economic_Activity
  [3] Labor Consumption
  [4] Fiscal Government
  [5] External Sector

🔢 Enter Aspect Number (0-5): 3

📊 ==================== MODEL ANALYSIS ====================
📝 Text  : "High interest rates continue to damp consumer demand, dragging down retail sales margins throughout the quarter"
🎯 Aspect: Labor Consumption
🏆 Result: 🔴 Negative (98.52% Confidence)

📈 Full Probability Breakdown Matrix:
  🟢 Positive  | ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ |   0.45%
  🔴 Negative  | █████████████████████████████░ |  98.52%
  🟡 Neutral   | ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ |   1.03%

--------------------------------------------------


In [10]:
import os
import re
import pandas as pd
import numpy as np

RAW_DATA_PATH = "/content/economic-news-sentiment/training/data/finbert_absa_training_ready.csv"
DOMAIN_LABELS = [
    "Monetary_Financial", "Inflation_Prices", "Real_Economic_Activity",
    "Labor_Consumption", "Fiscal_Government", "External_Sector"
]
SENTIMENT_MAP = {1: "🟢 Positive", 2: "🔴 Negative", 3: "🟡 Neutral"}

print("============== 📊 ECONOMIC SENTIMENT DATASET METRICS ==============")

# 1. Load Verification Lookups
if not os.path.exists(RAW_DATA_PATH):
    raise FileNotFoundError(f"❌ Missing dataset track at: {RAW_DATA_PATH}")

df = pd.read_csv(RAW_DATA_PATH)
total_raw_rows = len(df)

print(f"📁 Source File Path  : {RAW_DATA_PATH}")
print(f"📝 Raw Sentence Rows : {total_raw_rows}")
print(f"📐 Evaluated Domains : {len(DOMAIN_LABELS)}")
print("-" * 67)

# 2. Density / Sparsity Analysis (How many aspects trigger per sentence?)
# For each raw sentence, count how many domain columns have a value in [1, 2, 3]
active_aspects_per_row = df[DOMAIN_LABELS].isin([1, 2, 3]).sum(axis=1)
avg_density = active_aspects_per_row.mean()
zero_aspect_rows = (active_aspects_per_row == 0).sum()

print("🔍 ASPECT DENSITY PROFILE:")
print(f"  • Average aspects triggered per sentence: {avg_density:.2f}")
print(f"  • Sentences with 0 active aspects (dropped): {zero_aspect_rows} ({ (zero_aspect_rows/total_raw_rows)*100:.1f}%)")

# Calculate quick distribution of counts
counts_dist = active_aspects_per_row.value_counts().sort_index()
print("  • Distribution of aspect count per sentence:")
for num_aspects, count in counts_dist.items():
    bar = "█" * int((count/total_raw_rows) * 20)
    print(f"    [{num_aspects} aspects active]: {count:<5} | {bar}")

print("-" * 67)

# 3. Deep-Dive Matrix Exploration per Aspect Domain
print("📈 MATRICES BREAKDOWN PER MACRO DOMAIN LAYER:")
exploded_records = 0
matrix_summary = []

for aspect in DOMAIN_LABELS:
    clean_name = re.sub(r'_+', ' ', aspect).strip()
    col_data = df[aspect].value_counts()

    # Extract structural categories
    pos_count = col_data.get(1, 0)
    neg_count = col_data.get(2, 0)
    neu_count = col_data.get(3, 0)
    abstain_count = col_data.get(0, 0)

    active_total = pos_count + neg_count + neu_count
    exploded_records += active_total

    matrix_summary.append({
        "Aspect": clean_name,
        "Active": active_total,
        "Positive": pos_count,
        "Negative": neg_count,
        "Neutral": neu_count,
        "Abstain": abstain_count
    })

# Format structural matrix summary output into an aligned terminal visual chart
print(f"{'Target Macro Aspect':<24} | {'Active':<6} | {'🟢 Pos':<6} | {'🔴 Neg':<6} | {'🟡 Neu':<6} | {'💤 Skip':<6}")
print("—" * 67)
for row in matrix_summary:
    print(f"{row['Aspect']:<24} | {row['Active']:<6} | {row['Positive']:<6} | {row['Negative']:<6} | {row['Neutral']:<6} | {row['Abstain']:<6}")

print("-" * 67)
print(f"💥 Total Fine-Tuning Sequence Pairs Generated (Exploded Size): {exploded_records}")

# 4. Global Sentiment Bias Lookups
print("\n⚖️ GLOBAL EMBEDDED SENTIMENT BALANCE (Post-Explosion):")
global_pos = sum(r['Positive'] for r in matrix_summary)
global_neg = sum(r['Negative'] for r in matrix_summary)
global_neu = sum(r['Neutral'] for r in matrix_summary)

for label_name, label_count in [("🟢 Positive", global_pos), ("🔴 Negative", global_neg), ("🟡 Neutral", global_neu)]:
    ratio = label_count / exploded_records
    visual_bar = "█" * int(ratio * 30)
    print(f"  {label_name:<11} : {label_count:<5} ({ratio*100:5.2f}%) | {visual_bar}")

print("===================================================================")

============== 📊 ECONOMIC SENTIMENT DATASET METRICS ==============
📁 Source File Path  : /content/economic-news-sentiment/training/data/finbert_absa_training_ready.csv
📝 Raw Sentence Rows : 3224
📐 Evaluated Domains : 6
-------------------------------------------------------------------
🔍 ASPECT DENSITY PROFILE:
  • Average aspects triggered per sentence: 1.16
  • Sentences with 0 active aspects (dropped): 697 (21.6%)
  • Distribution of aspect count per sentence:
    [0 aspects active]: 697   | ████
    [1 aspects active]: 1528  | █████████
    [2 aspects active]: 800   | ████
    [3 aspects active]: 178   | █
    [4 aspects active]: 18    | 
    [5 aspects active]: 2     | 
    [6 aspects active]: 1     | 
-------------------------------------------------------------------
📈 MATRICES BREAKDOWN PER MACRO DOMAIN LAYER:
Target Macro Aspect      | Active | 🟢 Pos  | 🔴 Neg  | 🟡 Neu  | 💤 Skip
———————————————————————————————————————————————————————————————————
Monetary Financial       | 283  

In [11]:
import pandas as pd

RAW_DATA_PATH = "/content/economic-news-sentiment/training/data/finbert_absa_training_ready.csv"
DOMAIN_LABELS = [
    "Monetary_Financial", "Inflation_Prices", "Real_Economic_Activity",
    "Labor_Consumption", "Fiscal_Government", "External_Sector"
]

df = pd.read_csv(RAW_DATA_PATH)
total_rows = len(df)

mixed_sentiment_count = 0
multi_aspect_uniform_count = 0
single_aspect_count = 0
dropped_rows = 0

print("🔍 RUNNING MIXED SENTIMENT STRUCTURAL AUDIT...\n")

for idx, row in df.iterrows():
    # Gather all active sentiment values for this specific row (ignoring 0/Abstain)
    active_sentiments = [int(row[aspect]) for aspect in DOMAIN_LABELS if int(row[aspect]) in [1, 2, 3]]

    if len(active_sentiments) == 0:
        dropped_rows += 1
    elif len(active_sentiments) == 1:
        single_aspect_count += 1
    else:
        # If there are multiple aspects, check if the sentiment scores are different
        unique_sentiments = set(active_sentiments)
        if len(unique_sentiments) > 1:
            mixed_sentiment_count += 1
        else:
            multi_aspect_uniform_count += 1

# Display Analytics Results
print("=================== ⚖️ SENTIMENT COMPLEXITY PROFILE ===================")
print(f"📁 Total Sentences Evaluated : {total_rows}")
print("-" * 70)
print(f"💀 1. Completely Silent Rows (Dropped)  : {dropped_rows:<4} ({dropped_rows/total_rows*100:5.2f}%)")
print(f"🎯 2. Single-Aspect Rows                 : {single_aspect_count:<4} ({single_aspect_count/total_rows*100:5.2f}%)")
print(f"🤝 3. Multi-Aspect (Same Sentiment)     : {multi_aspect_uniform_count:<4} ({multi_aspect_uniform_count/total_rows*100:5.2f}%)")
print(f"⚡ 4. TRUE MIXED SENTIMENT ROWS         : {mixed_sentiment_count:<4} ({mixed_sentiment_count/total_rows*100:5.2f}%)")
print("======================================================================")

if mixed_sentiment_count == 0:
    print("\n🚨 WARNING: Zero mixed-sentiment sentences found! Your model has never\n"
          "seen a conflicting sentence. It literally doesn't know it's allowed to\n"
          "assign different sentiments to different aspects in the same pass.")
else:
    print(f"\n💡 Summary: Your dataset has {mixed_sentiment_count} true conflicting examples.\n"
          f"If this number is very low (e.g., less than 5% of your dataset), the model\n"
          f"will struggle to develop sharp, isolated attention maps.")

🔍 RUNNING MIXED SENTIMENT STRUCTURAL AUDIT...

=================== ⚖️ SENTIMENT COMPLEXITY PROFILE ===================
📁 Total Sentences Evaluated : 3224
----------------------------------------------------------------------
💀 1. Completely Silent Rows (Dropped)  : 697  (21.62%)
🎯 2. Single-Aspect Rows                 : 1528 (47.39%)
🤝 3. Multi-Aspect (Same Sentiment)     : 707  (21.93%)
⚡ 4. TRUE MIXED SENTIMENT ROWS         : 292  ( 9.06%)

💡 Summary: Your dataset has 292 true conflicting examples.
If this number is very low (e.g., less than 5% of your dataset), the model
will struggle to develop sharp, isolated attention maps.
